In [1]:
from importlib.metadata import version

import torch

print("torch:", torch.__version__)
print("CUDA dostupna:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("qdrant-client:", version("qdrant-client"))
print("open_clip:", version("open_clip_torch"))

torch: 2.14.0+cu126
CUDA dostupna: True
GPU: NVIDIA GeForce GTX 1050
qdrant-client: 1.19.0
open_clip: 3.3.0


In [2]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import torch
import torch.nn.functional as F

import open_clip
from PIL import Image

from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType

BACKEND_DIR = Path("..").resolve()
DATASET_DIR = BACKEND_DIR / "dataset"
TRAIN_DIR = DATASET_DIR / "train_ready"

QDRANT_URL = "http://localhost:6333"
COLLECTION = "plant_images"

MODEL_NAME = "hf-hub:imageomics/bioclip"
VECTOR_SIZE = 512
BATCH_SIZE = 64
NUM_WORKERS = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE =", DEVICE)

DEVICE = cuda


In [ ]:
MAPPING_CSV: Path = DATASET_DIR / "specie_id - scientific_name mapping.csv"

species_id_name_df: pd.DataFrame = pd.read_csv(MAPPING_CSV, dtype=str) 
species_id_name_dict: dict[str, str] = dict(zip(species_id_name_df["specie_id"], species_id_name_df["scientific_name"]))

In [ ]:
image_paths: list[Path] = sorted(TRAIN_DIR.glob("*/*.jpg"))

records: list[dict] = [] 
for i, path in enumerate(image_paths):
    specie_id = path.parent.name
    records.append({
        "id": i,
        "path": path,
        "payload": {
            "specie_id": specie_id,
            "scientific_name": species_id_name_dict[specie_id],
            "image_path": path.relative_to(DATASET_DIR).as_posix(),
        },
    })

print("Ukupno slika:", len(records))
print("Primer:", records[0]["id"], records[0]["payload"])

Ukupno slika: 136095
Primer: 0 {'specie_id': '0000', 'scientific_name': 'Lactuca virosa', 'image_path': 'train_ready/0000/01aca26dc4a0b0af7c55ecf84d8772179bf6fd6d.jpg'}


In [9]:
model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME)
model = model.to(DEVICE).eval() 

def image_preprocessing(path):
    with Image.open(path) as img:
        tensor=preprocess_val(img.convert("RGB"))
        return tensor
@torch.no_grad()
def embeddings_generator(batch):
    features = model.encode_image(batch.to(DEVICE))
    return F.normalize(features, dim=-1).cpu()


In [7]:
print(preprocess_val)

Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeConvertMode()
    MaybeToTensor()
    Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
)


In [12]:
client = QdrantClient(url=QDRANT_URL)

if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    client.create_payload_index(
        collection_name=COLLECTION,
        field_name="specie_id",
        field_schema=PayloadSchemaType.KEYWORD,
    )
    print("Kolekcija napravljena:", COLLECTION)
else:
    print("Kolekcija već postoji:", COLLECTION)

print("Tačaka u kolekciji:", client.count(COLLECTION).count)

Kolekcija već postoji: plant_images
Tačaka u kolekciji: 136095


In [ ]:
def load_preprocess_image(record):
    try:
        return image_preprocessing(record["path"])
    except Exception as e:
        tqdm.write(f"Preskačem {record['payload']['image_path']}: {e}")
        return None

start = client.count(COLLECTION).count
print(f"Počinjem od slike {start} / {len(records)}")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    for batch_start in tqdm(range(start, len(records), BATCH_SIZE), desc="Batch-evi"):
        batch:list[dict]= records[batch_start : batch_start + BATCH_SIZE]
        tensors = list(pool.map(load_preprocess_image, batch))
        
        record_tensor_pairs = [(record, tensor) for record, tensor in zip(batch, tensors) if tensor is not None]
        if not record_tensor_pairs:
            continue

        embeddings = embeddings_generator(torch.stack([t for _, t in record_tensor_pairs]))
        
        points = [
            PointStruct(id=record["id"], vector=embedding.tolist(), payload=record["payload"])
            for (record, _), embedding in zip(record_tensor_pairs, embeddings)
        ]
        client.upsert(collection_name=COLLECTION, points=points, wait=True)

print("Gotovo. Tačaka u kolekciji:", client.count(COLLECTION).count)

Počinjem od slike 0 / 136095


Batch-evi: 100%|██████████| 2127/2127 [3:05:44<00:00,  5.24s/it]  

Gotovo. Tačaka u kolekciji: 136095
